
# Research Watch updater for the combined codebase

Use this notebook from the **repo root** of the combined site.

What it does:
- processes **all normal `.docx` files** inside `weekly-inputs/`
- supports the new template field `Category: Academic / Industry / Company & Release`
- falls back to filename prefixes:
  - `academic-`
  - `industry-`
  - `company-release-`
- updates the root site files using the existing updater
- fixes the **AI Security Pulse** link lists in `ongoing-work.html`
- keeps the homepage **Latest note** tied to the newest processed file
- runs `scripts/sync_dual_site.py` at the end so `/dark/` is refreshed too

This notebook does **not** change theme, colors, CSS, or visual styling.


In [1]:

from pathlib import Path

REPO_ROOT = Path(".").resolve()
WEEKLY_INPUTS_DIR = REPO_ROOT / "weekly-inputs"
POSTS_DIR = REPO_ROOT / "posts"
ONGOING_HTML = REPO_ROOT / "ongoing-work.html"
INDEX_HTML = REPO_ROOT / "index.html"

REPLACE_DUPLICATE_POST_ID = True

TEMPLATE_DOCX_HINTS = {
    "weekly-research-watch-template.docx",
    "template.docx",
    "research-watch-template.docx",
    "ai_security_research_watch_template.docx",
}

print("Repo root:", REPO_ROOT)
print("weekly-inputs:", WEEKLY_INPUTS_DIR)
print("ongoing-work:", ONGOING_HTML)
print("index:", INDEX_HTML)


Repo root: C:\Users\Brojogopal.Sapui\Documents\personal_work\Git_workspace\AISecurityResearch-test
weekly-inputs: C:\Users\Brojogopal.Sapui\Documents\personal_work\Git_workspace\AISecurityResearch-test\weekly-inputs
ongoing-work: C:\Users\Brojogopal.Sapui\Documents\personal_work\Git_workspace\AISecurityResearch-test\ongoing-work.html
index: C:\Users\Brojogopal.Sapui\Documents\personal_work\Git_workspace\AISecurityResearch-test\index.html


In [2]:

import sys
import re
from datetime import datetime
from dataclasses import dataclass
from typing import Optional

SCRIPTS_DIR = REPO_ROOT / "scripts"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from docx import Document
from bs4 import BeautifulSoup

from research_watch_updater import (
    parse_weekly_docx,
    normalize_docx_data,
    update_site,
)


In [3]:

CATEGORY_ALIASES = {
    "academic": "Academic",
    "industry": "Industry",
    "company & release": "Company & Release",
    "company and release": "Company & Release",
    "companies & releases": "Company & Release",
    "companies and releases": "Company & Release",
    "company release": "Company & Release",
    "company-release": "Company & Release",
    "companies releases": "Company & Release",
    "ecosystem": "Company & Release",
}

FILENAME_PREFIX_TO_CATEGORY = {
    "academic-": "Academic",
    "industry-": "Industry",
    "company-release-": "Company & Release",
}

CATEGORY_TO_SECTION_ID = {
    "Academic": "academic-signals",
    "Industry": "industry-innovation",
    "Company & Release": "companies-releases",
}

@dataclass
class DocxMeta:
    path: Path
    title: str
    category: str
    explicit_date: Optional[datetime]
    sort_stamp: float

def _clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "").strip())

def normalize_category(raw: str) -> str:
    value = _clean_text(raw).strip(":- ").lower()
    return CATEGORY_ALIASES.get(value, "")

def category_from_filename(path: Path) -> str:
    stem_lower = path.stem.lower()
    for prefix, category in FILENAME_PREFIX_TO_CATEGORY.items():
        if stem_lower.startswith(prefix):
            return category
    return ""

def parse_date_value(raw: str) -> Optional[datetime]:
    raw = _clean_text(raw)
    if not raw:
        return None
    for fmt in ("%Y-%m-%d", "%Y-%m", "%d-%m-%Y", "%Y/%m/%d", "%Y/%m"):
        try:
            dt = datetime.strptime(raw, fmt)
            if fmt in ("%Y-%m", "%Y/%m"):
                return dt.replace(day=1)
            return dt
        except ValueError:
            pass
    m = re.search(r"(20\d{2})[-_/](\d{2})(?:[-_/](\d{2}))?", raw)
    if m:
        year = int(m.group(1))
        month = int(m.group(2))
        day = int(m.group(3) or 1)
        try:
            return datetime(year, month, day)
        except ValueError:
            return None
    return None

def extract_docx_meta(path: Path) -> DocxMeta:
    title = ""
    category = ""
    explicit_date = None

    doc = Document(path)
    lines = [_clean_text(p.text) for p in doc.paragraphs if _clean_text(p.text)]

    for line in lines[:20]:
        low = line.lower()
        if low.startswith("title:"):
            title = _clean_text(line.split(":", 1)[1])
        elif low.startswith("category:"):
            category = normalize_category(line.split(":", 1)[1])
        elif low.startswith("date"):
            explicit_date = parse_date_value(line.split(":", 1)[1])

    if not title:
        try:
            raw_data = parse_weekly_docx(path)
            title = _clean_text(raw_data.get("title", ""))
        except Exception:
            title = ""

    if not title:
        title = path.stem.replace("-", " ").replace("_", " ").strip().title()

    if not category:
        category = category_from_filename(path)

    if not category:
        title_low = title.lower()
        if any(word in title_low for word in ["paper", "survey", "benchmark", "arxiv", "conference"]):
            category = "Academic"
        elif any(word in title_low for word in ["startup", "launch", "release", "company", "funding", "platform"]):
            category = "Company & Release"
        else:
            category = "Industry"

    sort_stamp = (explicit_date.timestamp() if explicit_date else path.stat().st_mtime)
    return DocxMeta(path=path, title=title, category=category, explicit_date=explicit_date, sort_stamp=sort_stamp)

def is_valid_docx(path: Path) -> bool:
    name = path.name.lower()
    if path.suffix.lower() != ".docx":
        return False
    if name.startswith("~$"):
        return False
    if name in TEMPLATE_DOCX_HINTS:
        return False
    return True

docx_files = sorted([p for p in WEEKLY_INPUTS_DIR.glob("*.docx") if is_valid_docx(p)])
docx_meta = [extract_docx_meta(p) for p in docx_files]
docx_meta = sorted(docx_meta, key=lambda m: (m.sort_stamp, m.path.name.lower()))

print("DOCX files selected:")
for meta in docx_meta:
    d = meta.explicit_date.strftime("%Y-%m-%d") if meta.explicit_date else "mtime"
    print(f"- {meta.path.name} | {meta.category} | {d}")


DOCX files selected:
- 2026-03-stack-safeguard-pipelines.docx | Industry | mtime
- 2026-03-embodied-ai-security-governance.docx | Industry | mtime
- 2026-03-prompt-injection-defense.docx | Industry | mtime
- 2026-03-ai-assisted-sca-nn-recovery.docx | Industry | mtime
- 2026-03-belfort-encrypted-compute.docx | Industry | mtime
- Belfort_research_watch.docx | Industry | mtime
- Exein_research_watch.docx | Industry | mtime
- Qevlar AI_research_watch.docx | Industry | mtime
- RunSybil_research_watch.docx | Industry | mtime
- P0 Security_research_watch.docx | Industry | mtime
- Enclave_research_watch.docx | Industry | mtime
- Oligo Security_research_watch.docx | Industry | mtime
- Armada_research_watch.docx | Industry | mtime


In [4]:

results = []
by_category = {
    "Academic": [],
    "Industry": [],
    "Company & Release": [],
}

for meta in docx_meta:
    result = update_site(
        repo_root=REPO_ROOT,
        docx_path=meta.path,
        auto_pick_latest=False,
        replace_duplicate=REPLACE_DUPLICATE_POST_ID,
    )
    record = {
        "path": meta.path,
        "title": meta.title,
        "category": meta.category,
        "date": meta.explicit_date,
        "sort_stamp": meta.sort_stamp,
        "post_id": result.post_id,
        "post_path": Path(result.post_path),
    }
    results.append(record)
    by_category.setdefault(meta.category, []).append(record)

print("Processed files:", len(results))
for item in results:
    print(f"- {item['path'].name} -> {item['post_id']} [{item['category']}]")


Processed files: 13
- 2026-03-stack-safeguard-pipelines.docx -> 2026-03-stack-safeguard-pipelines [Industry]
- 2026-03-embodied-ai-security-governance.docx -> 2026-03-embodied-ai-security-governance [Industry]
- 2026-03-prompt-injection-defense.docx -> 2026-03-prompt-injection-defense [Industry]
- 2026-03-ai-assisted-sca-nn-recovery.docx -> 2026-03-ai-assisted-sca-nn-recovery [Industry]
- 2026-03-belfort-encrypted-compute.docx -> 2026-03-belfort-encrypted-compute [Industry]
- Belfort_research_watch.docx -> 2026-03-belfort-research-watch [Industry]
- Exein_research_watch.docx -> 2026-03-exein-research-watch [Industry]
- Qevlar AI_research_watch.docx -> 2026-03-qevlar-ai-research-watch [Industry]
- RunSybil_research_watch.docx -> 2026-03-runsybil-research-watch [Industry]
- P0 Security_research_watch.docx -> 2026-03-p0-security-research-watch [Industry]
- Enclave_research_watch.docx -> 2026-03-enclave-research-watch [Industry]
- Oligo Security_research_watch.docx -> 2026-03-oligo-securit

In [5]:

def _sort_records_desc(items):
    return sorted(items, key=lambda x: (x["sort_stamp"], x["path"].name.lower()), reverse=True)

for key in list(by_category.keys()):
    by_category[key] = _sort_records_desc(by_category[key])

results = _sort_records_desc(results)

def build_link_list_html(items):
    if not items:
        return '<div class="pulse-link-list"><a href="ongoing-work.html">Browse all research-watch notes</a></div>'
    links = ['<div class="pulse-link-list">']
    for item in items[:3]:
        text = item["title"]
        href = f'#{item["post_id"]}'
        links.append(f'<a href="{href}">{text}</a>')
    links.append("</div>")
    return "\n".join(links)

def replace_pulse_link_list(html_path: Path, section_id: str, items):
    html = html_path.read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "html.parser")
    section = soup.find(id=section_id)
    if section is None:
        print(f"Warning: section not found in {html_path.name}: {section_id}")
        return
    link_list = section.find("div", class_="pulse-link-list")
    if link_list is None:
        print(f"Warning: pulse-link-list not found in {html_path.name}: {section_id}")
        return

    new_fragment = BeautifulSoup(build_link_list_html(items), "html.parser")
    link_list.replace_with(new_fragment.div)
    html_path.write_text(str(soup), encoding="utf-8")
    print(f"Updated pulse-link-list in {html_path.name}: {section_id}")

replace_pulse_link_list(ONGOING_HTML, "academic-signals", by_category.get("Academic", []))
replace_pulse_link_list(ONGOING_HTML, "industry-innovation", by_category.get("Industry", []))
replace_pulse_link_list(ONGOING_HTML, "companies-releases", by_category.get("Company & Release", []))


Updated pulse-link-list in ongoing-work.html: academic-signals
Updated pulse-link-list in ongoing-work.html: industry-innovation
Updated pulse-link-list in ongoing-work.html: companies-releases


In [6]:

print("Newest homepage note should be:", results[0]["title"] if results else "None")
print("Generated posts:")
for item in results[:10]:
    print("-", item["post_path"].name)

print("")
print("Syncing root updates into /dark/ ...")


Newest homepage note should be: Armada Research Watch
Generated posts:
- 2026-03-armada-research-watch.html
- 2026-03-oligo-security-research-watch.html
- 2026-03-enclave-research-watch.html
- 2026-03-p0-security-research-watch.html
- 2026-03-runsybil-research-watch.html
- 2026-03-qevlar-ai-research-watch.html
- 2026-03-exein-research-watch.html
- 2026-03-belfort-research-watch.html
- 2026-03-belfort-encrypted-compute.html
- 2026-03-ai-assisted-sca-nn-recovery.html

Syncing root updates into /dark/ ...


In [7]:

!python scripts/sync_dual_site.py


Done: root content synced into /dark/ with dark theme preserved. Review, then commit + push.
